1.SETUP

Installing all the required libraries

In [1]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
)

***2.Load the dataset***


In [3]:
import os
from dotenv import load_dotenv
import kagglehub

load_dotenv()


token = os.getenv("KAGGLE_API_TOKEN")

path = kagglehub.competition_download('spaceship-titanic')

print("Path to competition files:", path)





c:\Users\Amith R\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: C:\Users\Amith R\.cache\kagglehub\competitions\spaceship-titanic


In [4]:
RANDOM_STATE = 42

In [5]:
import shutil

shutil.copytree(
    r"C:\Users\Amith R\.cache\kagglehub\competitions\spaceship-titanic", 
    r"C:\Users\Amith R\Music\ML\projects\datasets",
    dirs_exist_ok=True
)

print("Dataset copied successfully!")


Dataset copied successfully!


***3.EDA***

In [6]:
df = pd.read_csv("datasets/train.csv")


In [7]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [8]:

df.shape

(8693, 14)

In [9]:
df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [10]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


spliting some columns info fore more accuracy

In [11]:
df["Deck"] = df["Cabin"].str.split("/").str[0]
df["CabinNum"] = pd.to_numeric(
    df["Cabin"].str.split("/").str[1],
    errors="coerce"
)
df["Side"] = df["Cabin"].str.split("/").str[2]

Create total spending

In [12]:
df["TotalSpend"] = (
    df["RoomService"]
    + df["FoodCourt"]
    + df["ShoppingMall"]
    + df["Spa"]
    + df["VRDeck"]
)

Create GroupSize  look clearly we can see groups inside the id

In [13]:
df["GroupId"] = df["PassengerId"].str.split("_").str[0]

In [14]:
df["GroupSize"] = df.groupby("GroupId")["GroupId"].transform("count")

In [15]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,CabinNum,Side,TotalSpend,GroupId,GroupSize
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0.0,P,0.0,0001,1
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0.0,S,736.0,0002,1
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0.0,S,10383.0,0003,2
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0.0,S,5176.0,0003,2
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1.0,S,1091.0,0004,1


In [16]:
# duplicates
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()
print("Number of duplicate rows:", num_duplicates)

# (optional) drop duplicates if present
df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

Number of duplicate rows: 0
Shape after dropping duplicates: (8693, 20)


In [17]:
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,CabinNum,Side,TotalSpend,GroupId,GroupSize
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0.0,P,0.0,0001,1
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0.0,S,736.0,0002,1
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0.0,S,10383.0,0003,2
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0.0,S,5176.0,0003,2
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1.0,S,1091.0,0004,1




1.Removed the duplicates

2.Removed the id columns

3.Added extra sub-columns

**4. Data Preprocessing**

In [18]:
X = df.drop(columns=["Transported","Name","PassengerId","Cabin"])
y = df["Transported"]

In [19]:
X.head()

,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Deck,CabinNum,Side,TotalSpend,GroupId,GroupSize
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,B,0.0,P,0.0,0001,1
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,F,0.0,S,736.0,0002,1
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,A,0.0,S,10383.0,0003,2
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,A,0.0,S,5176.0,0003,2
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,F,1.0,S,1091.0,0004,1


In [20]:
y.head()

0    False
1     True
2    False
3    False
4     True
Name: Transported, dtype: bool

In [21]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

In [22]:
print("Train shape:", X_train.shape)

print("Test shape:", X_test.shape)

Train shape: (6954, 16)
Test shape: (1739, 16)


In [23]:
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# numerical features - preprocessing steps
# imputers can be added in numerical transformer
numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# categorical features - preprocessing steps
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# preprocessing pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Numerical features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum', 'TotalSpend', 'GroupSize']
Categorical features: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'GroupId']


In [24]:
baseline_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", DecisionTreeClassifier())
    ]
)

In [25]:
baseline_pipe.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
y_base_train_pred = baseline_pipe.predict(X_train)
y_base_test_pred = baseline_pipe.predict(X_test)

In [27]:
base_train_acc = round(accuracy_score(y_train, y_base_train_pred)*100, 2)
base_test_acc = round(accuracy_score(y_test, y_base_test_pred)*100, 2)

In [28]:
print("Baseline Train Accuracy:", base_train_acc)
print("Baseline Test Accuracy:", base_test_acc)

Baseline Train Accuracy: 99.97
Baseline Test Accuracy: 75.96


In [29]:
k = 5
cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

In [30]:
models = {
    "dt_balanced": Pipeline(
        [
            ("preprocess", preprocess),
            ("model", DecisionTreeClassifier(
                random_state=RANDOM_STATE,
                class_weight="balanced"
            ))
        ]
    ),

    "hgb_balanced": Pipeline(
        [
            ("preprocess", preprocess),
            ("model", HistGradientBoostingClassifier(
                random_state=RANDOM_STATE,
                class_weight="balanced"
            ))
        ]
    ),

    "rf_balanced": Pipeline(
        [
            ("preprocess", preprocess),
            ("model", RandomForestClassifier(
                random_state=RANDOM_STATE,
                class_weight="balanced_subsample"
            ))
        ]
    )
}

In [31]:
for name, m in models.items():

    pr_auc_scores = []

    for tr_idx, te_idx, in cv.split(X_train, y_train):
        X_tr, X_te = X_train.iloc[tr_idx], X_train.iloc[te_idx]
        y_tr, y_te = y_train.iloc[tr_idx], y_train.iloc[te_idx]    #creating folds  for each model we create 5 splits like this for loop runs 5 times for each model

        m.fit(X_tr, y_tr)
        pred_prob = m.predict_proba(X_te)[:, 1]

        pr_auc_scores.append(average_precision_score(y_te, pred_prob))

    print("Model name:", name)
    print("PR-AUC values:", pr_auc_scores)
    print("CV PR-AUC mean:", round(float(np.mean(pr_auc_scores)), 4))

Model name: dt_balanced
PR-AUC values: [0.7182381842456609, 0.7358009534550733, 0.7183771510161697, 0.6904323623712395, 0.7211463426852947]
CV PR-AUC mean: 0.7168
Model name: hgb_balanced
PR-AUC values: [0.909076547531136, 0.9177740975813355, 0.9201840261343099, 0.9011676723786213, 0.9148557770444097]
CV PR-AUC mean: 0.9126
Model name: rf_balanced
PR-AUC values: [0.8806023805067904, 0.898781689002105, 0.8943619707705578, 0.8792673234065916, 0.8988312942314701]
CV PR-AUC mean: 0.8904


In [32]:
hgb_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

In [33]:
param_dist = {
    "model__learning_rate": [0.01, 0.05, 0.1, 0.15, 0.2],
    "model__max_iter": [100, 200, 300, 400],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_leaf": [10, 20, 30, 50],
    "model__l2_regularization": [0, 0.1, 1, 10]
}

In [34]:
random_search = RandomizedSearchCV(
    estimator=hgb_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

In [35]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__l2_regularization': [0, 0.1, ...], 'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [None, 5, ...], 'model__max_iter': [100, 200, ...], ...}"
,n_iter,50
,scoring,'average_precision'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [36]:
print("Best CV PR-AUC:", round(random_search.best_score_, 4))
print("Best params:", random_search.best_params_)

Best CV PR-AUC: 0.9149
Best params: {'model__min_samples_leaf': 20, 'model__max_leaf_nodes': 15, 'model__max_iter': 400, 'model__max_depth': None, 'model__learning_rate': 0.05, 'model__l2_regularization': 0}


In [37]:
best_hgb = Pipeline(
    steps=[
        ("preprocess", preprocess),

        ("model", HistGradientBoostingClassifier(
            min_samples_leaf=random_search.best_params_["model__min_samples_leaf"],
            max_leaf_nodes=random_search.best_params_["model__max_leaf_nodes"],
            max_iter=random_search.best_params_["model__max_iter"],
            max_depth=random_search.best_params_["model__max_depth"],
            learning_rate=random_search.best_params_["model__learning_rate"],
            l2_regularization=random_search.best_params_["model__l2_regularization"],
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ]
)

In [38]:
best_hgb.fit(X_train, y_train)

print("Final model training completed!")

Final model training completed!


In [39]:
y_train_pred = best_hgb.predict(X_train)

In [40]:
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {round(train_acc*100, 2)} %")

Training Accuracy: 88.48 %


In [41]:
# Training predictions
y_train_pred = best_hgb.predict(X_train)

# Test predictions
y_test_pred = best_hgb.predict(X_test)

# Accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", round(train_accuracy * 100, 2), "%")
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")

Training Accuracy: 88.48 %
Test Accuracy: 81.02 %


In [52]:
def feature_engineering(df):
    df = df.copy()

    # Passenger group
    df["GroupId"] = df["PassengerId"].str.split("_").str[0]
    df["GroupSize"] = df.groupby("GroupId")["PassengerId"].transform("count")

    # Cabin
    df["Deck"] = df["Cabin"].str.split("/").str[0]
    df["CabinNum"] = pd.to_numeric(
        df["Cabin"].str.split("/").str[1],
        errors="coerce"
    )
    df["Side"] = df["Cabin"].str.split("/").str[2]

    # Total spending
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df["TotalSpend"] = df[spend_cols].sum(axis=1)

    return df

In [53]:
# Load test data
df_test = pd.read_csv("datasets/test.csv")

# Apply SAME feature engineering
df_test = feature_engineering(df_test)

# Remove columns not used by the model
X_kaggle_test = df_test.drop(
    columns=["Name", "PassengerId"]
)

print(X_kaggle_test.shape)
print(X_kaggle_test.columns)

(4277, 17)
Index(['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP',
       'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'GroupId',
       'GroupSize', 'Deck', 'CabinNum', 'Side', 'TotalSpend'],
      dtype='object')


In [54]:
test_predictions = best_hgb.predict(X_kaggle_test)

print("Number of predictions:", len(test_predictions))

Number of predictions: 4277


In [55]:
submission = pd.DataFrame({
    "PassengerId": df_test["PassengerId"],
    "Transported": test_predictions
})

submission.to_csv(
    "spaceship_submission2.csv",
    index=False
)
